# M0 · physical — hand playground

The real **BrainCo Revo 2** right hand. Simulation twin:
[`../simulation/hand_playground.ipynb`](../simulation/hand_playground.ipynb) — start there if
you have not driven the joint space yet.

**Stack:** `bc_stark_sdk` (BrainCo's async SDK) → Modbus-RTU over RS-485 → a private asyncio
loop on a background thread → `ipywidgets`. Backend code: [`real_hand.py`](real_hand.py).
No pydrake anywhere in this stack.

**Shared with the simulation stack:** only [`../hand_model.py`](../hand_model.py) and the pose
files it reads and writes.

### Before you power the hand
- Nothing inside the fingers; the hand clear of the table and of the arm.
- `SPEED` low (200–400 of 1000) to start.
- Fingers that stall keep pulling current — watch the current readout, and hit **OPEN (stop)**
  if a finger is loaded but not moving.
- **Interrupting the kernel does not stop the hand.** It keeps holding its last commanded pose.
  Use **OPEN (stop)**, or run the shutdown cell at the bottom.

### Setup on the robot PC
```bash
pip install bc-stark-sdk
ls /dev/ttyUSB*                   # find the RS-485 adapter
sudo usermod -aG dialout $USER    # then log out and back in
```

In [ ]:
import sys
from pathlib import Path

# Locate src/m0 whether the kernel started in this folder, in the repo root, or anywhere between.
# Matched by content, not by folder name, so it survives m0/M0 casing differences between
# a case-insensitive mac and the case-sensitive robot PC.
_here = [Path.cwd(), *Path.cwd().parents]
_candidates = [*_here, *(p / "src" / name for p in _here for name in ("m0", "M0"))]
M0_DIR = next((p for p in _candidates if (p / "hand_model.py").exists()), None)
if M0_DIR is None:
    raise RuntimeError(f"could not find src/m0 from {Path.cwd()}; open this notebook from its own folder")
for _path in (M0_DIR, M0_DIR / "physical"):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

import time

import numpy as np

from hand_model import FINGERS, load_poses
from real_hand import RIGHT_HAND_ID, RealHand, RealHandUI, libstark

PORT = None              # None -> auto-detect the adapter; else e.g. "/dev/ttyUSB0"
SLAVE_ID = RIGHT_HAND_ID # 0x7F right hand, 0x7E left hand
BAUDRATE = 460800        # ignored while auto-detecting
SPEED = 400              # 0..1000

print("bc_stark_sdk:", "available" if libstark is not None else "NOT INSTALLED (pip install bc-stark-sdk)")

## Connect

`connect()` auto-detects port, baud rate and slave id, sets the hand to normalized units, and
starts the 50 Hz streaming thread. Streaming does not move anything yet — the hand stays
disarmed until you press the ARM button.

In [ ]:
hand = RealHand(port=PORT, slave_id=SLAVE_ID, baudrate=BAUDRATE, speed=SPEED)
hand.connect()

print(hand.status())

## Control

Press **ARM** to let the streaming thread write to the hand; everything below it is live from
that moment. `OPEN (stop)` is the panic button.

The counters at the bottom of the panel are the diagnostic: *slider updates* proves the widget
events reach Python, *frames sent* proves the streaming thread is writing to the serial port.
If poses do not move the hand while both counters climb, the problem is downstream (power,
wiring, slave id) — not the notebook.

In [ ]:
ui = RealHandUI(hand)
ui

## First moves

Run these one at a time, with a hand on the power switch. The first one is the important one:
it confirms that the SDK's finger order really is
`thumb, thumb_aux, index, middle, ring, pinky` on *this* hand.

In [ ]:
assert hand.armed, "press ARM in the panel above first"

for finger in FINGERS:
    print(f"closing {finger.name}")
    hand.move_to({finger.name: 0.8}, duration=0.6, steps=20)
    time.sleep(0.5)

hand.open_hand()

In [ ]:
# Replay the poses designed in simulation.
poses = load_poses()
hand.play([(name, poses[name]) for name in ["open", "pinch", "open", "lego_pinch", "open"]], hold=0.6)

### Commanded vs measured

The interesting column is `error`. A finger that stops short of its command while drawing
current has hit something — that is the signal M1 will use to know a brick is held.

In [ ]:
result = hand.hold_and_measure(poses["lego_pinch"])

print(f"{'finger':10s} {'cmd':>6s} {'meas':>6s} {'err':>7s} {'mA':>7s}")
for finger in FINGERS:
    i = finger.sdk_index
    current = result["currents_mA"][i] if len(result["currents_mA"]) > i else float("nan")
    print(f"{finger.name:10s} {result['commanded'][i]:6.2f} {result['measured'][i]:6.2f} "
          f"{result['error'][i]:7.2f} {current:7.0f}")

hand.open_hand()

## Shutdown

Always leave the hand open and release the port — an open `bc_stark_sdk` context blocks the next
process from claiming `/dev/ttyUSB*`.

In [ ]:
hand.close()   # opens the hand, stops streaming, closes the serial port

## Next

[`grasp_poses.ipynb`](grasp_poses.ipynb) — replay the whole simulated library, find the closure
where the fingers actually grip a brick, and write `../poses_measured.json`.

## Troubleshooting

| Symptom | Cause |
|---|---|
| `bc_stark_sdk not installed` | `pip install bc-stark-sdk` in the `int2026` env (Linux only). |
| `auto_detect` finds nothing | Adapter not enumerated (`ls /dev/ttyUSB*`), hand unpowered, or A/B lines swapped. |
| `Permission denied: /dev/ttyUSB0` | Not in `dialout` yet, or you have not logged out since `usermod`. Quick test: `sudo chmod 666 /dev/ttyUSB0`. |
| Port busy / open fails | A previous kernel still holds it. Run `hand.close()` there, or restart that kernel. |
| Counters climb, hand still | Wrong slave id (0x7F right / 0x7E left), or the hand is not powered. |
| Sliders do nothing at all | `ipywidgets` missing from the kernel's env — `conda activate int2026` before `jupyter lab`. |
| A finger buzzes and stalls | It is loaded. Open the hand, then lower that entry in `hand_model.POSITION_CAP`. |